# Rajuk Image Level Classifier — Bagged SVM on Deep Features

**Model:** a **Support Vector Machine** ensemble built with **bagging (bootstrap aggregating)**.

Images cannot be fed to an SVM directly (a 224x224 RGB image is a 150,528-dimensional
vector — far too high-dimensional and too pixel-aligned for a kernel machine). So the
pipeline is:

```
image → frozen ResNet18 (ImageNet, no training) → 512-d feature vector
      → StandardScaler → PCA
      → BaggingClassifier( SVC(rbf) x N_ENSEMBLE )   ← the actual classifier
```

The ResNet18 is used **only as a fixed feature extractor** — its weights are never
updated. All learning happens in the SVMs.

**Pipeline stages:**
1. **Leakage-free split** by unique subject ID — a subject's augmented copies never
   straddle train/val/test.
2. **Hard-example mining** — validation images the previous run got wrong are folded
   back into training.
3. **Deep feature extraction** — one deterministic pass plus `TRAIN_AUG_PASSES`
   augmented passes, so augmentation still enlarges the training set (each augmented
   view becomes an extra feature row).
4. **StandardScaler + PCA** — SVMs need scaled inputs; PCA cuts 512 dims to the
   components holding `PCA_COMPONENTS` of the variance (denoising → lower variance).
5. **Grid search** over `C` and `gamma` with leakage-free grouped CV.
6. **Bagging** — `N_ENSEMBLE` SVMs, each fit on a bootstrap resample (drawn *with
   replacement*) and a random `MAX_FEATURES_FRAC` subset of features. Their
   probabilities are averaged (**soft voting**).
7. **Out-of-bag (OOB) scoring** — each member is scored on the ~37 % of rows its
   bootstrap missed, so the validation set is never touched while building the ensemble.
8. **Flip TTA** — predictions averaged over the image and its mirror.

**Why bagging an SVM?** A single RBF-SVM on ~1k images has high **variance**: change the
sample slightly and the support vectors — and the decision boundary — move. Averaging
`N_ENSEMBLE` SVMs trained on independent bootstrap resamples cancels the errors that are
independent across members, so the variance term of the error drops roughly like 1/N,
while the bias stays put.

## 1. Local Setup (no Google Drive needed)

This notebook runs **locally on your NVIDIA GPU**. Images are read directly from
the project folders on disk — no Colab / Drive mount required.

In [ ]:
# Running locally — no Google Drive mount needed.
# Image folders are read straight from disk (paths set in the Configuration cell).
print('Running locally — no Drive mount required.')

## 2. Import Libraries

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T

# ── The model itself: SVM + bagging (all learning happens here) ───────────────
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, log_loss)
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV)
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_GROUP_KFOLD = True
except ImportError:          # scikit-learn < 1.0
    HAS_GROUP_KFOLD = False

import joblib

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}   (GPU is used only for feature extraction)')

## 3. Configuration

In [ ]:
# ── Paths (LOCAL) ───────────────────────────────────────────────────────────────
# Root folder of the Final Project on this machine.
DRIVE_BASE    = Path(r'D:\Downloads\DUCSE Documents\4-1\Mine\CSE-4125 Introduction to Machine Learning\Lab\Final Project\taining files')
AUGMENTED_DIR = DRIVE_BASE / 'Augmented_raw_data'
REAL_DIR      = DRIVE_BASE / 'raw_data' / 'final train image'
XLSX_PATH     = DRIVE_BASE / 'Rajuk_ML_DataSet.xlsx'
CSV_PATH      = DRIVE_BASE / 'Real ML_DataSet.csv'

# Where trained artifacts / plots are written (keep them next to this notebook).
OUTPUT_DIR    = Path.cwd()

# Sanity-check the paths so failures are obvious up front.
for label, pth in [('Augmented dir', AUGMENTED_DIR), ('Real images dir', REAL_DIR)]:
    print(f'  {label:16s}: {pth}  ->  {"OK" if pth.exists() else "MISSING!"}')
if not (XLSX_PATH.exists() or CSV_PATH.exists()):
    print('  WARNING: neither the .xlsx nor the .csv label file was found!')

# ── Data settings ──────────────────────────────────────────────────────────────
CLASS_NAMES   = ['negative', 'moderate', 'positive']
ALLOWED_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp'}
IMAGE_SIZE    = (224, 224)

# ── Feature extraction (frozen ResNet18 — NOT trained) ────────────────────────
FEATURE_BATCH    = 64     # batch size for the single forward pass over each split
TRAIN_AUG_PASSES = 2      # extra augmented passes over the train images → more rows
                          # (0 = deterministic pass only; each pass ≈ +1x train rows)

# ── SVM hyper-parameters ──────────────────────────────────────────────────────
SVM_KERNEL     = 'rbf'    # RBF handles the non-linear class boundaries
SVM_C          = 10.0     # inverse regularisation — HIGH C = low bias / high variance
SVM_GAMMA      = 'scale'  # RBF width — HIGH gamma = wigglier boundary / high variance
PCA_COMPONENTS = 0.95     # keep components explaining 95 % of variance (None disables)
CLASS_WEIGHT   = 'balanced'   # counteracts class imbalance (replaces the CNN's weighted loss)

# ── Bagging (bootstrap aggregating) — the variance reducer ───────────────────
N_ENSEMBLE        = 15    # number of SVMs; each fit on its own bootstrap resample
BOOTSTRAP_FRAC    = 1.0   # bootstrap size as a fraction of the train set (WITH replacement)
MAX_FEATURES_FRAC = 0.8   # random feature subset per member → extra member diversity
USE_OOB           = True  # score each member on its out-of-bag rows
TTA_ENABLED       = True  # average predictions over the image and its mirror

# ── Hyper-parameter search ────────────────────────────────────────────────────
RUN_GRID_SEARCH = True    # set False to skip and use SVM_C / SVM_GAMMA as-is
GRID_C          = [1.0, 10.0, 100.0]
GRID_GAMMA      = ['scale', 0.01, 0.001]
KFOLD_SPLITS    = 3
N_JOBS          = -1      # parallel SVM fits (-1 = all cores)

# ── Split fractions (applied to unique REAL subject IDs) ──────────────────────
# Target ratio  →  TRAIN : TEST : VALIDATION  =  70 : 20 : 10  (%)
# Splitting is done on unique subject IDs (leakage-free); each subject's augmented
# copies follow it into the same split, so the file-level ratio tracks the subject
# ratio closely (augmentation is ~uniform per subject).
TEST_FRACTION = 0.20
VAL_FRACTION  = 0.10

# ── Learning-curve dataset steps ──────────────────────────────────────────────
DATASET_STEPS  = [200, 500, 800, 1100]
CURVE_ENSEMBLE = 5        # smaller ensemble for the learning-curve sweep (speed)

# ImageNet normalisation (required for the pretrained ResNet feature extractor)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('\nConfiguration loaded.')
print(f'  Image size     : {IMAGE_SIZE}')
print(f'  Features       : frozen ResNet18 penultimate layer (512-d)')
print(f'  Aug passes     : {TRAIN_AUG_PASSES}  (train rows ≈ {1 + TRAIN_AUG_PASSES}x images)')
print(f'  Classifier     : SVC(kernel={SVM_KERNEL}, C={SVM_C}, gamma={SVM_GAMMA})')
print(f'  PCA            : {PCA_COMPONENTS}  |  class_weight: {CLASS_WEIGHT}')
print(f'  Bagging        : {N_ENSEMBLE} members  |  max_samples={BOOTSTRAP_FRAC}  '
      f'max_features={MAX_FEATURES_FRAC}')
print(f'  OOB scoring    : {USE_OOB}  |  Flip TTA: {TTA_ENABLED}')
print(f'  Grid search    : {RUN_GRID_SEARCH}  |  K-fold splits: {KFOLD_SPLITS}')
print(f'  Val fraction   : {VAL_FRACTION}  |  Test fraction : {TEST_FRACTION}')

## 4. Load Labels from Excel / CSV

In [ ]:
def normalize_header(name):
    return ''.join(ch for ch in str(name).lower() if ch.isalnum())

def find_column(df, target_normalized):
    for col in df.columns:
        if normalize_header(col) == target_normalized:
            return col
    return None

def normalize_level(value):
    if pd.isna(value):
        return None
    text = str(value).strip().lower()
    mapping = {'p': 'positive', 'positive': 'positive',
               'm': 'moderate',  'moderate': 'moderate',
               'n': 'negative',  'negative': 'negative'}
    return mapping.get(text)

def load_label_mapping(df, level_col, class_names):
    label_lookup = {name: idx for idx, name in enumerate(class_names)}
    label_map = {}
    for row_index, value in enumerate(df[level_col].tolist(), start=1):
        normalized = normalize_level(value)
        if normalized is None:
            continue
        label_map[row_index] = label_lookup[normalized]
    return label_map

if XLSX_PATH.exists():
    df = pd.read_excel(XLSX_PATH)
    source_path = XLSX_PATH
elif CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    source_path = CSV_PATH
else:
    raise FileNotFoundError(
        f'No dataset file found at {DRIVE_BASE}. '
        'Expected Rajuk_ML_DataSet.xlsx or Real ML_DataSet - Sheet1.csv'
    )

level_col = (
    find_column(df, 'levelpositivemoderatenegative')
    or find_column(df, 'level')
)
if level_col is None:
    raise ValueError('Required level column not found in the dataset.')

label_map = load_label_mapping(df, level_col, CLASS_NAMES)
print(f'Loaded {len(df):,} rows  |  {len(label_map):,} valid labels  |  source: {source_path.name}')

## 5. Collect Image Paths & Leakage-Free Split

Split is done by **unique subject ID** (real images only) so that no subject's augmented
copy can appear in val/test. Augmented images are assigned to training only.

In [ ]:
def parse_image_index(stem):
    match = re.match(r'img(\d+)', stem.lower())
    return int(match.group(1)) if match else None

def collect_all_raw(base_dir, label_map):
    paths, labels = [], []
    skip_idx = skip_lbl = 0
    if not base_dir.exists():
        return paths, labels, skip_idx, skip_lbl
    for p in base_dir.rglob('*'):
        if not (p.is_file() and p.suffix.lower() in ALLOWED_EXTS):
            continue
        idx = parse_image_index(p.stem.strip())
        if idx is None:
            skip_idx += 1; continue
        lbl = label_map.get(idx)
        if lbl is None:
            skip_lbl += 1; continue
        paths.append(p)
        labels.append(lbl)
    return paths, labels, skip_idx, skip_lbl

aug_paths,  aug_labels,  si_a, sl_a = collect_all_raw(AUGMENTED_DIR, label_map)
real_paths, real_labels, si_r, sl_r = collect_all_raw(REAL_DIR,      label_map)

print(f'Augmented images         : {len(aug_paths):,}')
print(f'Real images              : {len(real_paths):,}')
print(f'Skipped (no index)       : {si_a + si_r}')
print(f'Skipped (no label match) : {sl_a + sl_r}')

# ── Leakage-free split by unique subject ID ────────────────────────────────────
unique_indices     = sorted(set(
    parse_image_index(p.stem.strip()) for p in real_paths
    if parse_image_index(p.stem.strip()) is not None
))
unique_labels_list = [label_map[i] for i in unique_indices]
all_uq             = np.arange(len(unique_indices))

trainval_uq, test_uq = train_test_split(
    all_uq, test_size=TEST_FRACTION, random_state=SEED, stratify=unique_labels_list
)
val_rel = VAL_FRACTION / (1.0 - TEST_FRACTION)
train_uq, val_uq = train_test_split(
    trainval_uq, test_size=val_rel, random_state=SEED,
    stratify=np.array(unique_labels_list)[trainval_uq]
)

train_img_ids = set(np.array(unique_indices)[train_uq])
val_img_ids   = set(np.array(unique_indices)[val_uq])
test_img_ids  = set(np.array(unique_indices)[test_uq])

train_paths_final, train_labels_final = [], []
val_paths_final,   val_labels_final   = [], []
test_paths_final,  test_labels_final  = [], []

# Augmented → each subject's aug copies follow it into its split
for p, lbl in zip(aug_paths, aug_labels):
    img_id = parse_image_index(p.stem.strip())
    if img_id in train_img_ids:
        train_paths_final.append(p); train_labels_final.append(lbl)
    elif img_id in val_img_ids:
        val_paths_final.append(p);   val_labels_final.append(lbl)
    elif img_id in test_img_ids:
        test_paths_final.append(p);  test_labels_final.append(lbl)

# Real → correct split
for p, lbl in zip(real_paths, real_labels):
    img_id = parse_image_index(p.stem.strip())
    if img_id in train_img_ids:
        train_paths_final.append(p); train_labels_final.append(lbl)
    elif img_id in val_img_ids:
        val_paths_final.append(p);   val_labels_final.append(lbl)
    elif img_id in test_img_ids:
        test_paths_final.append(p);  test_labels_final.append(lbl)

N_total = len(train_paths_final) + len(val_paths_final) + len(test_paths_final)
print(f'\nLeakage-free split:')
print(f'  Train (real + aug) : {len(train_paths_final):,}  ({100*len(train_paths_final)/N_total:.1f} %)')
print(f'  Val   (real + aug) : {len(val_paths_final):,}  ({100*len(val_paths_final)/N_total:.1f} %)')
print(f'  Test  (real + aug) : {len(test_paths_final):,}  ({100*len(test_paths_final)/N_total:.1f} %)')
print(f'  Total accounted    : {N_total:,}  (of {len(aug_paths)+len(real_paths):,} raw — diff = {len(aug_paths)+len(real_paths)-N_total} skipped)')

val_counts  = np.bincount(val_labels_final,  minlength=len(CLASS_NAMES))
test_counts = np.bincount(test_labels_final, minlength=len(CLASS_NAMES))
print('\nValidation-set class distribution:')
for name, cnt in zip(CLASS_NAMES, val_counts):
    print(f'  {name:10s}: {cnt:,}')
print('\nTest-set class distribution:')
for name, cnt in zip(CLASS_NAMES, test_counts):
    print(f'  {name:10s}: {cnt:,}')

all_labels_combined = train_labels_final + val_labels_final + test_labels_final
counts = np.bincount(all_labels_combined, minlength=len(CLASS_NAMES))
print('\nClass distribution (total):')
for name, cnt in zip(CLASS_NAMES, counts):
    print(f'  {name:10s}: {cnt:,}')

## 5b. Hard-Example Mining — Inject Misclassified Validation Images into TRAIN

The previous run exported every image the model got wrong on the validation set to
`val_misclassified.csv`. Here we feed those *hard examples* back into the **training
set** so the network gets a second chance to learn them:

* **all** misclassified **raw / real** images  → moved into TRAIN
* **some** of their **augmented** copies (a per-subject sample controlled by
  `AUG_KEEP_FRACTION`) → moved into TRAIN

Matched images are pulled out of the val / test lists and appended to the train lists
*before* the datasets are cached, so the change flows through the entire pipeline
(weighted loss, k-fold, learning curves, final report).

> **Note:** because only *some* augmented copies of a hard subject are moved while the
> rest stay in validation, those specific hard subjects can appear in both splits. This
> is a deliberate trade-off of hard-example mining (the goal is to *learn the hard
> cases*), not a clean random-split guarantee.

In [ ]:
# ── Hard-example mining: move misclassified validation images into TRAIN ───
from collections import defaultdict

MISCLASSIFIED_CSV = OUTPUT_DIR / 'val_misclassified.csv'
AUG_KEEP_FRACTION = 0.5     # fraction of each subject's misclassified AUG copies -> train

def _norm_path(p):
    return os.path.normcase(os.path.normpath(str(p)))

if not MISCLASSIFIED_CSV.exists():
    print(f'⚠  {MISCLASSIFIED_CSV.name} not found — skipping hard-example mining.')
else:
    hard_df = pd.read_csv(MISCLASSIFIED_CSV)
    print(f'Loaded {len(hard_df):,} misclassified rows from {MISCLASSIFIED_CSV.name}')

    # Split the misclassified rows into raw (real) vs augmented
    real_mask = hard_df['image_type'].astype(str).str.lower().eq('real')
    hard_real_paths = {_norm_path(p) for p in hard_df.loc[real_mask, 'file_path']}

    # Group augmented misclassified paths by subject id, then sample "some" per subject
    aug_by_subject = defaultdict(list)
    for _, r in hard_df.loc[~real_mask].iterrows():
        aug_by_subject[r['image_id']].append(_norm_path(r['file_path']))

    rng = np.random.RandomState(SEED)
    hard_aug_selected = set()
    for iid, plist in aug_by_subject.items():
        k = max(1, int(round(len(plist) * AUG_KEEP_FRACTION)))
        chosen = rng.choice(len(plist), size=min(k, len(plist)), replace=False)
        hard_aug_selected.update(plist[j] for j in chosen)

    move_set = hard_real_paths | hard_aug_selected

    # Pull matched paths out of val / test; collect them to append to train
    def _extract(paths, labels):
        keep_p, keep_l, moved = [], [], []
        for p, l in zip(paths, labels):
            if _norm_path(p) in move_set:
                moved.append((p, l))
            else:
                keep_p.append(p); keep_l.append(l)
        return keep_p, keep_l, moved

    val_paths_final,  val_labels_final,  moved_val  = _extract(val_paths_final,  val_labels_final)
    test_paths_final, test_labels_final, moved_test = _extract(test_paths_final, test_labels_final)

    already_in_train = {_norm_path(p) for p in train_paths_final}
    n_already = len(move_set & already_in_train)

    for p, l in moved_val + moved_test:
        train_paths_final.append(p); train_labels_final.append(l)

    n_aug_total = sum(len(v) for v in aug_by_subject.values())
    print(f'  Misclassified RAW images         : {len(hard_real_paths):,}  (all → train)')
    print(f'  Misclassified AUG images (total) : {n_aug_total:,}')
    print(f'  AUG copies selected for train    : {len(hard_aug_selected):,}  '
          f'(≈{AUG_KEEP_FRACTION:.0%} per subject)')
    print(f'  Moved into TRAIN  (from val)     : {len(moved_val):,}')
    print(f'  Moved into TRAIN  (from test)    : {len(moved_test):,}')
    print(f'  Already in TRAIN (split changed) : {n_already:,}')

    N_total = len(train_paths_final) + len(val_paths_final) + len(test_paths_final)
    print('\nSplit after hard-example mining:')
    print(f'  Train : {len(train_paths_final):,}  ({100*len(train_paths_final)/N_total:.1f} %)')
    print(f'  Val   : {len(val_paths_final):,}  ({100*len(val_paths_final)/N_total:.1f} %)')
    print(f'  Test  : {len(test_paths_final):,}  ({100*len(test_paths_final)/N_total:.1f} %)')

## 6. 📷 Sample Images — Data Preview

In [ ]:
SAMPLES_PER_CLASS = 3
label_to_real_paths = {i: [] for i in range(len(CLASS_NAMES))}
for p, lbl in zip(real_paths, real_labels):
    label_to_real_paths[lbl].append(p)

rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(
    len(CLASS_NAMES), SAMPLES_PER_CLASS,
    figsize=(SAMPLES_PER_CLASS * 3.5, len(CLASS_NAMES) * 3.5)
)
fig.suptitle('Sample Images  (one row per class)', fontsize=14, fontweight='bold', y=1.01)
border_colours = ['#d62728', '#ff7f0e', '#2ca02c']

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    candidates = label_to_real_paths[cls_idx]
    chosen = rng.choice(len(candidates),
                        size=min(SAMPLES_PER_CLASS, len(candidates)),
                        replace=False)
    for col, path_idx in enumerate(chosen):
        ax = axes[cls_idx][col]
        img = Image.open(candidates[path_idx]).convert('RGB').resize(IMAGE_SIZE)
        ax.imshow(np.array(img))
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(border_colours[cls_idx])
            spine.set_linewidth(3)
        if col == 0:
            ax.set_ylabel(cls_name.upper(), fontsize=11, fontweight='bold',
                          color=border_colours[cls_idx], rotation=90, labelpad=8)

plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sample images displayed.')

## 7. Image Cache & Transforms

The images are cached in RAM once, then read repeatedly by the feature extractor.

**Why keep augmentation for an SVM?** An SVM has no epochs, so it can't see a fresh
augmentation each pass. Instead we run the extractor over the training images
`TRAIN_AUG_PASSES` extra times with random augmentation on — each augmented view becomes
an **additional feature row**. That enlarges and diversifies the training matrix, which
is the SVM equivalent of on-the-fly augmentation. Val/test get resize + normalise only.

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transform = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class RajukDataset(Dataset):
    """
    Caches images as PIL uint8 objects (NOT float32 tensors).
    RAM cost: N * H * W * 3 * 1 byte  (12x cheaper than float32 tensors)

    For 2700 images at 224x224:
      PIL uint8   → 0.41 GB  ✅ safe on 12.7 GB RAM
      float32     → 1.63 GB  ❌ what caused the crash before

    Transforms applied fresh each __getitem__ → different augmentation every epoch.
    """

    def __init__(self, image_paths, labels, transform):
        self.labels    = labels
        self.transform = transform
        n = len(image_paths)
        ram_mb = n * IMAGE_SIZE[0] * IMAGE_SIZE[1] * 3 / 1e6
        print(f'  Caching {n:,} PIL images at {IMAGE_SIZE} … (est. {ram_mb:.0f} MB)')
        self.images = []
        for p in image_paths:
            with Image.open(p) as img:
                # Store as resized uint8 PIL — smallest possible in-memory form
                self.images.append(img.convert('RGB').resize(IMAGE_SIZE, Image.BILINEAR))
        print(f'  Done. All {n:,} images in RAM.')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Fresh transform each call = on-the-fly augmentation every epoch
        return self.transform(self.images[idx]), self.labels[idx]


if len(train_paths_final) == 0:
    raise ValueError('No labeled images found. Check folder paths and label mapping.')

print('Building datasets (PIL uint8 RAM cache at 224x224):')
train_dataset = RajukDataset(train_paths_final, train_labels_final, train_transform)
val_dataset   = RajukDataset(val_paths_final,   val_labels_final,   val_test_transform)
test_dataset  = RajukDataset(test_paths_final,  test_labels_final,  val_test_transform)

total_mb = (len(train_dataset)+len(val_dataset)+len(test_dataset)) * 224*224*3 / 1e6
print(f'\nDatasets ready — Train: {len(train_dataset):,}  '
      f'Val: {len(val_dataset):,}  Test: {len(test_dataset):,}')
print(f'Total image RAM: ~{total_mb:.0f} MB  (out of 12,700 MB available)')


## 8. Loader Settings

In [ ]:
pin = torch.cuda.is_available()
# On Windows + Jupyter, DataLoader worker processes (num_workers>0) frequently hang
# because the in-RAM PIL cache must be pickled to each spawned worker. The images are
# already cached in memory, so workers add little benefit — use 0 for reliability.
NW = 0

print(f'Train : {len(train_dataset):,} images')
print(f'Val   : {len(val_dataset):,} images')
print(f'Test  : {len(test_dataset):,} images')
print(f'\npin_memory={pin}  num_workers={NW}')

## 9. Frozen ResNet18 Feature Extractor

An RBF-SVM on raw pixels is hopeless here: 150,528 dimensions, ~1k training images, and
a kernel that measures raw pixel distance (two photos of the same building shifted a few
pixels look maximally far apart). So we borrow ImageNet's learned visual vocabulary.

`resnet18(pretrained)` with its classification head removed maps each image to a
**512-d descriptor** encoding edges, textures and shapes. **Every weight is frozen** —
there is no backprop, no optimiser, no epochs in this notebook. The network is a fixed
function applied once per image, exactly like HOG or SIFT would be, just far better.
All decision-making is done by the SVMs downstream.

In [ ]:
class ResNetFeatureExtractor(nn.Module):
    """Pretrained ResNet18 with the classifier head replaced by identity.

    forward(x) -> (B, 512) descriptors. All parameters are frozen and the module is
    permanently in eval() mode, so BatchNorm uses its ImageNet running statistics
    and the output for a given image is deterministic.
    """

    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.out_dim = backbone.fc.in_features      # 512 for ResNet18
        backbone.fc  = nn.Identity()
        for p in backbone.parameters():
            p.requires_grad = False
        self.model = backbone.eval()

    def forward(self, x):
        return self.model(x)


extractor = ResNetFeatureExtractor().to(device).eval()

n_params   = sum(p.numel() for p in extractor.parameters())
n_trainable = sum(p.numel() for p in extractor.parameters() if p.requires_grad)
print(f'Feature extractor : ResNet18 (ImageNet pretrained)')
print(f'Output dimension  : {extractor.out_dim}')
print(f'Total parameters  : {n_params:,}')
print(f'Trainable params  : {n_trainable:,}   ← frozen: nothing is learned here')

## 10. Extract Feature Matrices

Each split is pushed through the extractor once to build a plain `(n_samples, 512)`
design matrix — from here on this is an ordinary tabular classification problem.

| Matrix | Rows | Transform | Used for |
|--------|------|-----------|----------|
| `X_train` | images x (1 + `TRAIN_AUG_PASSES`) | deterministic + augmented | **fitting** the bagged SVM |
| `X_train_det` | images | deterministic | measuring clean training error |
| `X_val` / `X_test` | images | deterministic | evaluation |
| `*_flip` | same | mirrored | flip-TTA |

`groups_train` carries each row's **subject ID** so cross-validation can keep every copy
of a subject inside one fold — without it, an augmented copy in the CV-train fold and its
original in the CV-val fold would leak and inflate the score.

In [ ]:
class _ViewDataset(Dataset):
    """A re-transformed view of an already-cached RajukDataset.

    Bypasses the base dataset's own transform so the same RAM cache can be read with
    the augmenting transform (extra train rows) or the deterministic one (evaluation),
    optionally mirrored for TTA.
    """

    def __init__(self, base_ds, transform, flip=False):
        self.base, self.transform, self.flip = base_ds, transform, flip

    def __len__(self):
        return len(self.base.labels)

    def __getitem__(self, i):
        img = self.base.images[i]
        if self.flip:
            img = ImageOps.mirror(img)
        return self.transform(img), self.base.labels[i]


@torch.no_grad()
def extract_features(base_ds, transform, flip=False, label=''):
    """Run the frozen extractor over a dataset → (X, y) numpy arrays."""
    loader = DataLoader(_ViewDataset(base_ds, transform, flip),
                        batch_size=FEATURE_BATCH, shuffle=False,
                        num_workers=NW, pin_memory=pin)
    feats, labs = [], []
    for X, y in loader:
        feats.append(extractor(X.to(device)).cpu().numpy())
        labs.append(np.asarray(y))
    X = np.concatenate(feats).astype(np.float32)
    y = np.concatenate(labs).astype(np.int64)
    print(f'  {label:<34} → {X.shape[0]:>5,} x {X.shape[1]}')
    return X, y


def subject_ids(paths):
    """Subject ID per file — the grouping key that keeps CV leakage-free."""
    return np.array([parse_image_index(p.stem.strip()) or -1 for p in paths])


print('Extracting deep features (single forward pass per view):')

# TRAIN — deterministic pass (clean training-error measurement) …
X_train_det, y_train_det = extract_features(
    train_dataset, val_test_transform, label='train (deterministic)')
groups_det = subject_ids(train_paths_final)

# … plus augmented passes, each adding a fresh set of rows for the SVM to fit on
X_parts, y_parts, g_parts = [X_train_det], [y_train_det], [groups_det]
for a in range(TRAIN_AUG_PASSES):
    torch.manual_seed(SEED + 1000 + a)     # different augmentation stream per pass
    Xa, ya = extract_features(train_dataset, train_transform,
                              label=f'train (augmented pass {a+1})')
    X_parts.append(Xa); y_parts.append(ya); g_parts.append(groups_det)

X_train      = np.vstack(X_parts)
y_train      = np.concatenate(y_parts)
groups_train = np.concatenate(g_parts)

# VAL / TEST — deterministic, plus mirrored views for TTA
X_val,  y_val  = extract_features(val_dataset,  val_test_transform, label='val')
X_test, y_test = extract_features(test_dataset, val_test_transform, label='test')

if TTA_ENABLED:
    X_val_flip,  _ = extract_features(val_dataset,  val_test_transform, flip=True,
                                      label='val  (flipped, TTA)')
    X_test_flip, _ = extract_features(test_dataset, val_test_transform, flip=True,
                                      label='test (flipped, TTA)')
    X_train_det_flip, _ = extract_features(train_dataset, val_test_transform, flip=True,
                                           label='train (flipped, TTA)')
else:
    X_val_flip = X_test_flip = X_train_det_flip = None

# Kept for compatibility with the CNN notebook's downstream cells
train_labels_arr = np.array(train_labels_final, dtype=np.int64)
class_counts     = np.bincount(y_train, minlength=len(CLASS_NAMES))

print(f'\nFeature matrices ready.')
print(f'  X_train      : {X_train.shape}   ← fed to the bagged SVM')
print(f'  X_train_det  : {X_train_det.shape}   ← clean training-error view')
print(f'  X_val        : {X_val.shape}')
print(f'  X_test       : {X_test.shape}')
print(f'  Unique train subjects : {len(np.unique(groups_train)):,}')

print('\nClass counts in the training matrix:')
for name, cnt in zip(CLASS_NAMES, class_counts):
    print(f'  {name:10s}: {cnt:,} rows  ({100*cnt/len(y_train):.1f} %)')
print(f"\nclass_weight='{CLASS_WEIGHT}' makes each SVM re-weight C per class inversely "
      f'to these counts,\nso the majority class cannot simply swallow the boundary.')

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 11. SVM Pipeline & Hyper-parameter Search

Each ensemble member is a small `Pipeline`, not a bare `SVC`:

1. **`StandardScaler`** — an RBF kernel measures Euclidean distance, so a feature with a
   larger numeric range would silently dominate the kernel. Scaling is not optional.
2. **`PCA`** — projects 512 dims onto the components carrying `PCA_COMPONENTS` of the
   variance. Dropping the low-variance tail removes noise directions the SVM would
   otherwise fit → **lower variance** and a faster fit.
3. **`SVC(rbf)`** — the classifier.

Both preprocessing steps live *inside* the pipeline so that when bagging refits a member
on a bootstrap resample, the scaler and PCA are refit **on that resample only** — no
statistics leak across members, and each member is a genuinely independent model.

`C` and `gamma` are the bias–variance dials: large `C` and large `gamma` buy a tighter
fit (low bias, high variance). The search uses **grouped** CV so a subject's augmented
copies never straddle a fold.

In [ ]:
# sklearn renamed BaggingClassifier's base estimator argument in 1.2
try:
    BaggingClassifier(estimator=SVC(), n_estimators=1)
    EST_KW = 'estimator'
except TypeError:
    EST_KW = 'base_estimator'


def make_svm_pipeline(C=None, gamma=None, probability=True):
    """Scale → PCA → SVC. Refit end-to-end by every bagging member."""
    steps = [('scale', StandardScaler())]
    if PCA_COMPONENTS:
        steps.append(('pca', PCA(n_components=PCA_COMPONENTS, whiten=True,
                                 random_state=SEED)))
    steps.append(('svm', SVC(
        kernel=SVM_KERNEL,
        C=SVM_C if C is None else C,
        gamma=SVM_GAMMA if gamma is None else gamma,
        class_weight=CLASS_WEIGHT,
        probability=probability,   # needed for soft voting; costs an internal Platt CV
        random_state=SEED,
    )))
    return Pipeline(steps)


def make_bagged_svm(C=None, gamma=None, n_estimators=None, oob=None):
    """Bagging aggregator over independent SVM pipelines (soft voting)."""
    kwargs = {
        EST_KW          : make_svm_pipeline(C, gamma),
        'n_estimators'  : N_ENSEMBLE if n_estimators is None else n_estimators,
        'max_samples'   : BOOTSTRAP_FRAC,
        'bootstrap'     : True,             # ← draw WITH replacement = the bootstrap
        'max_features'  : MAX_FEATURES_FRAC,
        'bootstrap_features': False,
        'oob_score'     : USE_OOB if oob is None else oob,
        'n_jobs'        : N_JOBS,
        'random_state'  : SEED,
    }
    return BaggingClassifier(**kwargs)


def make_cv(n_splits):
    """Grouped CV when available → a subject's copies stay inside one fold."""
    if HAS_GROUP_KFOLD:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED), True
    print('  (StratifiedGroupKFold unavailable — falling back to StratifiedKFold; '
          'CV scores may be optimistic.)')
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED), False


# ── Grid search over C / gamma on a SINGLE svm (bagging is added afterwards) ──
best_C, best_gamma = SVM_C, SVM_GAMMA

if RUN_GRID_SEARCH:
    cv, grouped = make_cv(KFOLD_SPLITS)
    # probability=False here: Platt scaling is expensive and irrelevant to accuracy ranking
    search = GridSearchCV(
        make_svm_pipeline(probability=False),
        param_grid={'svm__C': GRID_C, 'svm__gamma': GRID_GAMMA},
        scoring='accuracy', cv=cv, n_jobs=N_JOBS, refit=False, verbose=1,
    )
    fit_kw = {'groups': groups_train} if grouped else {}
    print(f'Grid search: {len(GRID_C)} x {len(GRID_GAMMA)} combos x {KFOLD_SPLITS} folds '
          f'on {X_train.shape[0]:,} rows …')
    search.fit(X_train, y_train, **fit_kw)

    best_C     = search.best_params_['svm__C']
    best_gamma = search.best_params_['svm__gamma']

    res = pd.DataFrame(search.cv_results_)[
        ['param_svm__C', 'param_svm__gamma', 'mean_test_score', 'std_test_score']
    ].sort_values('mean_test_score', ascending=False)
    res.columns = ['C', 'gamma', 'mean_cv_acc', 'std']
    print('\nGrid-search results (best first):')
    print(res.to_string(index=False))
    print(f'\nBest params : C={best_C}, gamma={best_gamma}  '
          f'→ CV acc {search.best_score_:.4f}')
else:
    print(f'Grid search skipped — using C={best_C}, gamma={best_gamma} from the config.')

## 11b. K-Fold Cross-Validation of the Bagged SVM

The grid search tuned a *single* SVM. Here the full **bagged** ensemble is cross-validated
with the chosen hyper-parameters, giving a mean ± std estimate of its accuracy that does
not consume the validation set. The spread across folds is itself a variance read-out:
a tight spread means the model is stable under resampling.

In [ ]:
cv, grouped = make_cv(KFOLD_SPLITS)
splitter = (cv.split(X_train, y_train, groups_train) if grouped
            else cv.split(X_train, y_train))

fold_val_accs = []

for fold, (tr, va) in enumerate(splitter):
    print(f'\n{"─"*58}')
    print(f'  Fold {fold+1}/{KFOLD_SPLITS}   |   train={len(tr):,}   val={len(va):,}')
    print(f'{"─"*58}')

    # oob=False: OOB scoring is meaningless on a CV sub-fit and just costs time
    fold_bag = make_bagged_svm(best_C, best_gamma, oob=False)
    fold_bag.fit(X_train[tr], y_train[tr])

    acc = accuracy_score(y_train[va], fold_bag.predict(X_train[va]))
    fold_val_accs.append(acc)
    print(f'  → Fold {fold+1} accuracy: {acc:.4f}')

print(f'\nCross-val accuracies : {[f"{a:.4f}" for a in fold_val_accs]}')
print(f'Mean CV accuracy     : {np.mean(fold_val_accs):.4f} ± {np.std(fold_val_accs):.4f}')

## 12. Bagging — Bootstrap-Aggregated SVM Ensemble → Evaluate on Val + Test

This is the headline model. `BaggingClassifier` does the following, `N_ENSEMBLE` times:

1. Draw a **bootstrap sample** of the training rows — same size, drawn *with replacement*,
   so ~63 % of rows appear (some more than once) and ~37 % are left **out-of-bag**.
2. Draw a random `MAX_FEATURES_FRAC` subset of the feature columns (random subspace) —
   extra decorrelation between members.
3. Refit the whole pipeline (scaler → PCA → SVC) on that resample.

At predict time the members' `predict_proba` outputs are averaged — **soft voting**.
Errors that are independent across members cancel; the shared, systematic part (the bias)
survives. That is precisely the variance term of the error dropping ~1/N.

**OOB score** is the honest self-estimate: every training row is predicted using only the
members that never saw it. The validation set stays untouched until the cell below reports on it.

In [ ]:
print(f'Fitting {N_ENSEMBLE} SVMs on bootstrap resamples of {X_train.shape[0]:,} rows '
      f'(C={best_C}, gamma={best_gamma}) …')

bagged_svm = make_bagged_svm(best_C, best_gamma)
bagged_svm.fit(X_train, y_train)
final_model = bagged_svm          # canonical name used by the cells below

print(f'Done. Members: {len(bagged_svm.estimators_)}')
if USE_OOB and hasattr(bagged_svm, 'oob_score_'):
    print(f'Out-of-bag accuracy : {bagged_svm.oob_score_:.4f}   '
          f'(each row scored only by members that never saw it)')

n_sv = [est.named_steps["svm"].n_support_.sum() for est in bagged_svm.estimators_]
print(f'Support vectors per member : mean {np.mean(n_sv):.0f}  '
      f'(min {min(n_sv)}, max {max(n_sv)})  of {X_train.shape[0]:,} rows')


# ── Prediction helpers (soft voting + flip TTA) ──────────────────────────────
LABELS = list(range(len(CLASS_NAMES)))

def bag_proba(bag, X, X_flip=None):
    """Averaged member probabilities, optionally averaged over the mirrored view too."""
    p = bag.predict_proba(X)
    if TTA_ENABLED and X_flip is not None:
        p = 0.5 * (p + bag.predict_proba(X_flip))
    return p


def eval_split(bag, X, y, X_flip=None):
    """→ (log_loss, accuracy, preds, true). Mirrors the CNN notebook's evaluate() signature."""
    p     = bag_proba(bag, X, X_flip)
    preds = p.argmax(1)
    return (log_loss(y, p, labels=LABELS), accuracy_score(y, preds),
            list(preds), list(y))


def member_probas(bag, X):
    """Per-member probabilities, expanded to the full class space."""
    out = []
    for est, feat in zip(bag.estimators_, bag.estimators_features_):
        p   = est.predict_proba(X[:, feat])
        cls = np.asarray(est.classes_)
        if len(cls) == len(CLASS_NAMES):
            out.append(p)
        else:                       # a bootstrap missed a class — pad the gap
            full = np.zeros((X.shape[0], len(CLASS_NAMES)))
            full[:, cls] = p
            out.append(full)
    return out


val_loss,  val_acc,  val_preds,  val_true  = eval_split(bagged_svm, X_val,  y_val,  X_val_flip)
test_loss, test_acc, test_preds, test_true = eval_split(bagged_svm, X_test, y_test, X_test_flip)

print(f'\n{"="*50}')
print(f'  BAGGED SVM — {len(bagged_svm.estimators_)} members'
      + (', flip-TTA' if TTA_ENABLED else ''))
print(f'  Validation — Loss: {val_loss:.4f}   Acc: {val_acc:.4f}')
print(f'  Test       — Loss: {test_loss:.4f}   Acc: {test_acc:.4f}')
print(f'{"="*50}')

# ── Variance-reduction check: one SVM vs the ensemble ────────────────────────
single_svm = make_svm_pipeline(best_C, best_gamma)
single_svm.fit(X_train, y_train)
s_val_acc = accuracy_score(y_val, single_svm.predict(X_val))

member_accs = [accuracy_score(y_val, p.argmax(1)) for p in member_probas(bagged_svm, X_val)]

print(f'\nSingle SVM (all data, no bagging) — Val Acc : {s_val_acc:.4f}')
print(f'Individual members  — mean Val Acc          : {np.mean(member_accs):.4f} '
      f'± {np.std(member_accs):.4f}  (spread = the variance bagging averages away)')
print(f'Bagged ensemble     — Val Acc               : {val_acc:.4f}')
print(f'Ensemble gain over the mean member          : '
      f'{val_acc - np.mean(member_accs):+.4f}  (variance removed by bagging)')
print(f'Ensemble gain over a single full-data SVM   : {val_acc - s_val_acc:+.4f}')

## 13. Classification Report & Confusion Matrix

In [ ]:
print('Test Set — Classification Report')
print('─' * 50)
print(classification_report(test_true, test_preds, target_names=CLASS_NAMES))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (preds, true, title, cmap) in zip(
    axes,
    [
        (val_preds,  val_true,  'Confusion Matrix — Validation Set', 'OrRd'),
        (test_preds, test_true, 'Confusion Matrix — Test Set',       'Blues'),
    ]
):
    cm = confusion_matrix(true, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 13b. Final Error Report — Train / Validation / Test

A single consolidated report of the **bagged SVM** (`final_model`). Training error is
measured on `X_train_det` — the deterministic, un-augmented feature pass — so it reflects
the model's true fit rather than augmentation noise. `error = 1 − accuracy`.

In [ ]:
tr_loss_f,  tr_acc_f,  _, _ = eval_split(final_model, X_train_det, y_train_det, X_train_det_flip)
val_loss_f, val_acc_f, _, _ = eval_split(final_model, X_val,  y_val,  X_val_flip)
te_loss_f,  te_acc_f,  _, _ = eval_split(final_model, X_test, y_test, X_test_flip)

report_rows = [
    ('Training',   len(y_train_det), tr_loss_f,  tr_acc_f,  1 - tr_acc_f),
    ('Validation', len(y_val),       val_loss_f, val_acc_f, 1 - val_acc_f),
    ('Test',       len(y_test),      te_loss_f,  te_acc_f,  1 - te_acc_f),
]

print('=' * 68)
print(f'  FINAL ERROR REPORT  —  model = bagged SVM '
      f'({len(final_model.estimators_)} members, C={best_C}, gamma={best_gamma})')
print('=' * 68)
print(f'  {"Split":<12}{"Samples":>9}{"Loss":>12}{"Accuracy":>12}{"Error":>12}')
print('  ' + '-' * 64)
for name, n, loss, acc, err in report_rows:
    print(f'  {name:<12}{n:>9,}{loss:>12.4f}{acc:>11.2%}{err:>12.2%}')
print('=' * 68)
if USE_OOB and hasattr(final_model, 'oob_score_'):
    print(f'  Out-of-bag error (honest, no val set used) : {1 - final_model.oob_score_:.2%}')
    print('=' * 68)

# ── Bar chart of the three error rates ──────────────────────────────
names  = [r[0] for r in report_rows]
errors = [r[4] for r in report_rows]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(names, errors, color=colors, width=0.6, edgecolor='black', linewidth=0.7)
for b, err in zip(bars, errors):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005,
            f'{err:.2%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Error rate  (1 − accuracy)', fontsize=11)
ax.set_title('Bagged SVM — Train / Validation / Test Error', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(errors) * 1.25 + 0.02)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('final_error_report.png', dpi=130, bbox_inches='tight')
plt.show()

## 12b. Misclassified Validation Examples — Manual Error Analysis

The cells below identify **every misclassified example in the validation set**, export
them to a CSV, and display them as an image grid. Each misclassified example is also
assigned to a group member (~20 per member) so the team can fill in the
*"possible reason for misclassification"* sheet required by the report.

> ⚠️ Run cell 12 (evaluation) first so `val_preds`, `val_true`, and `val_paths_final`
> are populated.

In [ ]:
# ── Identify all misclassified validation images ──────────────────────────────
# Features were extracted with shuffle=False → val rows align 1-to-1 with val_paths_final
val_true_arr = np.array(val_true)
val_pred_arr = np.array(val_preds)

assert len(val_paths_final) == len(val_true_arr), \
    'Length mismatch — re-run section 12 (fit the bagged SVM) before this cell.'

mis_mask = val_true_arr != val_pred_arr
mis_idx  = np.where(mis_mask)[0]

print(f'Validation set size  : {len(val_true_arr):,}')
print(f'Correctly classified : {int((~mis_mask).sum()):,}')
print(f'Misclassified        : {int(mis_mask.sum()):,}  ({100*mis_mask.mean():.1f} %)')

# Resolve augmented dir once for type detection
aug_dir_resolved = AUGMENTED_DIR.resolve() if AUGMENTED_DIR.exists() else None

N_MEMBERS = 4   # ← change to your actual group size

rows = []
for i in mis_idx:
    path   = val_paths_final[i]
    img_id = parse_image_index(path.stem.strip())
    try:
        is_aug = aug_dir_resolved is not None and \
                 path.resolve().is_relative_to(aug_dir_resolved)
    except Exception:
        is_aug = 'aug' in str(path).lower()
    rows.append({
        'val_index'       : int(i),
        'image_id'        : img_id,
        'file_name'       : path.name,
        'image_type'      : 'augmented' if is_aug else 'real',
        'true_label'      : CLASS_NAMES[val_true_arr[i]],
        'pred_label'      : CLASS_NAMES[val_pred_arr[i]],
        'error_type'      : f'{CLASS_NAMES[val_true_arr[i]]} → {CLASS_NAMES[val_pred_arr[i]]}',
        'file_path'       : str(path),
        'reason_category' : '',   # fill: A=ambiguous B=quality C=occlusion D=lighting E=label_error F=distribution G=imbalance H=other
        'possible_reason' : '',
        'suggested_fix'   : '',
    })

mis_df = pd.DataFrame(rows)

if len(mis_df) > 0:
    per_member = int(np.ceil(len(mis_df) / max(1, N_MEMBERS)))
    mis_df['assigned_member'] = (np.arange(len(mis_df)) // per_member) + 1

# ── Print every misclassified filename ───────────────────────────────────────
print('\n' + '─' * 72)
print(f'  {"#":>4}  {"file_name":<40} {"type":<10}  {"true":>10} → pred')
print('─' * 72)
for rank, row in mis_df.iterrows():
    print(f'  {int(rank)+1:>4}  {row["file_name"]:<40} {row["image_type"]:<10}  '
          f'{row["true_label"]:>10} → {row["pred_label"]}')
print('─' * 72)

print(f'\nError-type breakdown:')
print(mis_df['error_type'].value_counts().to_string())

print(f'\nBreakdown by image type (real vs augmented):')
print(mis_df.groupby(['image_type', 'error_type']).size().rename('count').to_string())

# ── Save CSV ──────────────────────────────────────────────────────────────────
mis_csv = OUTPUT_DIR / 'val_misclassified.csv'
mis_df.to_csv(mis_csv, index=False)
print(f'\nCSV saved → {mis_csv}')
print(f'Columns    : {list(mis_df.columns)}')

mis_df

In [ ]:
# ── Visualise misclassified validation images ─────────────────────────────────
# Shows up to MAX_DISPLAY images (all errors go to the CSV; grid is just a preview).
MAX_DISPLAY = 60   # enough for ~4 members × 15 examples each

if len(mis_idx) == 0:
    print('No misclassified validation examples — nothing to display.')
else:
    display_idx = mis_idx[:MAX_DISPLAY]
    n     = len(display_idx)
    cols  = 5
    rows_ = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * 3.0, rows_ * 3.3))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis('off')

    for j, i in enumerate(display_idx):
        ax  = axes[j]
        img = val_dataset.images[i]
        ax.imshow(np.array(img))
        ax.set_xticks([]); ax.set_yticks([])
        ax.axis('on')
        row = mis_df[mis_df['val_index'] == i]
        member = int(row['assigned_member'].iloc[0]) if len(row) > 0 else '?'
        ax.set_title(
            f'idx {i} | id {parse_image_index(val_paths_final[i].stem.strip())}\n'
            f'True: {CLASS_NAMES[val_true_arr[i]]}\n'
            f'Pred: {CLASS_NAMES[val_pred_arr[i]]}  (M{member})',
            fontsize=8, color='#d62728', fontweight='bold'
        )
        for spine in ax.spines.values():
            spine.set_edgecolor('#d62728'); spine.set_linewidth(2)

    total_errors = len(mis_idx)
    shown = len(display_idx)
    fig.suptitle(
        f'Misclassified Validation Examples  '
        f'(showing {shown} of {total_errors} total — see val_misclassified.csv for all)',
        fontsize=12, fontweight='bold', y=1.005
    )
    plt.tight_layout()
    plt.savefig('val_misclassified.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Total misclassified : {total_errors}')
    print(f'Displayed           : {shown}  (first {MAX_DISPLAY})')
    print(f'Full list saved to  : val_misclassified.csv')

## 14. Learning Curves — Error vs. Dataset Size

| Marker | Meaning |
|--------|---------|
| 🔵 Blue circle | **Training error** |
| 🟠 Orange triangle | **Validation error** |
| 🟢 Green square | **Test error** |


In [ ]:
def train_and_measure(n_train: int):
    """Fit a bagged SVM on a stratified subset of n_train rows → (train, val, test) loss."""
    actual_n = min(n_train, len(X_train))
    if actual_n < len(CLASS_NAMES) * 2:
        return None

    if actual_n < len(X_train):
        sub_idx, _ = train_test_split(
            np.arange(len(X_train)), train_size=actual_n,
            random_state=SEED, stratify=y_train
        )
    else:
        sub_idx = np.arange(len(X_train))

    # Smaller ensemble + no OOB/TTA here — this sweep refits the model once per size
    bag = make_bagged_svm(best_C, best_gamma, n_estimators=CURVE_ENSEMBLE, oob=False)
    bag.fit(X_train[sub_idx], y_train[sub_idx])

    tr_loss = log_loss(y_train[sub_idx], bag.predict_proba(X_train[sub_idx]), labels=LABELS)
    vl_loss = log_loss(y_val,  bag.predict_proba(X_val),  labels=LABELS)
    te_loss = log_loss(y_test, bag.predict_proba(X_test), labels=LABELS)
    return tr_loss, vl_loss, te_loss


curve_sizes, curve_train, curve_val, curve_test = [], [], [], []

sweep_steps = [n for n in DATASET_STEPS if n < len(X_train)] + [len(X_train)]

for n in sweep_steps:
    print(f'  Fitting on n={n:,} …', end=' ', flush=True)
    result = train_and_measure(n)
    if result is None:
        print('skipped (too few samples).'); continue
    tr_l, vl_l, te_l = result
    curve_sizes.append(n)
    curve_train.append(tr_l)
    curve_val.append(vl_l)
    curve_test.append(te_l)
    print(f'train={tr_l:.4f}  val={vl_l:.4f}  test={te_l:.4f}')

print('\nLearning-curve data collected.')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(curve_sizes, curve_train, color='#1f77b4', linewidth=1.8, linestyle='--', alpha=0.6)
ax.plot(curve_sizes, curve_val,   color='#ff7f0e', linewidth=1.8, linestyle='--', alpha=0.6)
ax.plot(curve_sizes, curve_test,  color='#2ca02c', linewidth=1.8, linestyle='--', alpha=0.6)

ax.scatter(curve_sizes, curve_train, marker='o', s=100,
           color='#1f77b4', zorder=5, label='Training Error')
ax.scatter(curve_sizes, curve_val,   marker='^', s=110,
           color='#ff7f0e', zorder=5, label='Validation Error')
ax.scatter(curve_sizes, curve_test,  marker='s', s=100,
           color='#2ca02c', zorder=5, label='Test Error')

for s in curve_sizes:
    ax.axvline(s, color='grey', linewidth=0.4, linestyle=':', alpha=0.5)

ax.set_xlabel('Number of Training Samples', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('Learning Curve — Train / Validation / Test Error\n'
             '(3 dots per dataset size: ● Train  ▲ Val  ■ Test)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.9)
ax.set_xticks(curve_sizes)
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('learning_curve.png', dpi=130, bbox_inches='tight')
plt.show()
print('Learning curve saved.')

## 14b. Bias–Variance Analysis

Using the learning-curve measurements above we visualise the classic
**bias–variance decomposition** of the model's error:

* **Bias**  →  the error floor the *training* curve settles at (how well the model
  can fit the data at all — a high floor ⇒ under-fitting).
* **Variance**  →  the **gap** between the validation and training error
  (how much performance drops on unseen data — a large gap ⇒ over-fitting).

The shaded band is the variance gap; the dashed line marks the bias floor. The right
panel stacks the two components to show how they add up to the total validation error.

In [ ]:
# ── Bias–Variance visualisation (derived from the learning-curve sweep) ────
if len(curve_sizes) < 1:
    print('No learning-curve data — run the learning-curve cells (section 14) first.')
else:
    sizes = np.asarray(curve_sizes, dtype=float)
    tr    = np.asarray(curve_train, dtype=float)
    vl    = np.asarray(curve_val,   dtype=float)

    bias_floor   = float(tr[-1])                    # train error at largest size ≈ bias
    variance_gap = float(vl[-1] - tr[-1])           # val − train gap    ≈ variance
    var_disp     = max(variance_gap, 0.0)           # clamp for the stacked bar

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5.2),
                                   gridspec_kw={'width_ratios': [1.6, 1]})

    # Left: train vs val error, variance gap shaded, bias floor marked
    axL.plot(sizes, tr, 'o-', color='#1f77b4', linewidth=2, label='Training error (→ bias)')
    axL.plot(sizes, vl, '^-', color='#ff7f0e', linewidth=2, label='Validation error')
    axL.fill_between(sizes, tr, vl, color='#ff7f0e', alpha=0.15,
                     label='Variance (val − train gap)')
    axL.axhline(bias_floor, color='#1f77b4', linestyle='--', linewidth=1.2, alpha=0.8)
    axL.annotate('Bias floor', xy=(sizes[0], bias_floor),
                 xytext=(sizes[0], bias_floor + 0.03),
                 fontsize=10, color='#1f77b4', fontweight='bold')
    axL.set_xlabel('Number of Training Samples', fontsize=11)
    axL.set_ylabel('Cross-Entropy Loss (error)', fontsize=11)
    axL.set_title('Bias–Variance: Train vs Validation Error', fontsize=12, fontweight='bold')
    axL.set_xticks(sizes)
    axL.tick_params(axis='x', rotation=30)
    axL.legend(fontsize=10, framealpha=0.9)
    axL.grid(axis='y', linestyle='--', alpha=0.4)

    # Right: decomposition of the final error into bias + variance
    axR.bar(0, bias_floor, color='#1f77b4', edgecolor='black', linewidth=0.7, width=0.55)
    axR.bar(1, var_disp,   color='#ff7f0e', edgecolor='black', linewidth=0.7, width=0.55)
    axR.bar(2, bias_floor, color='#1f77b4', edgecolor='black', linewidth=0.7, width=0.55)
    axR.bar(2, var_disp, bottom=bias_floor, color='#ff7f0e',
            edgecolor='black', linewidth=0.7, width=0.55)
    for x, val in [(0, bias_floor), (1, var_disp), (2, bias_floor + var_disp)]:
        axR.text(x, val + 0.01, f'{val:.3f}', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')
    axR.set_xticks([0, 1, 2])
    axR.set_xticklabels(['Bias\n(train err)', 'Variance\n(gap)', 'Total\n(val err)'])
    axR.set_ylabel('Cross-Entropy Loss', fontsize=11)
    axR.set_title('Error Decomposition (largest train size)', fontsize=12, fontweight='bold')
    axR.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig('bias_variance.png', dpi=130, bbox_inches='tight')
    plt.show()

    print(f'Bias  (training-error floor) : {bias_floor:.4f}')
    print(f'Variance (val − train gap)    : {variance_gap:.4f}')
    print(f'Total validation error       : {float(vl[-1]):.4f}')

## 15. Model Complexity & Ensemble Size

Two views of the bias–variance trade-off that are specific to a bagged SVM, replacing the
CNN's per-epoch training history (an SVM has no epochs — it solves its optimisation in
one shot):

* **Left — error vs `C`.** `C` is the SVM's complexity dial. Small `C` = a heavily
  regularised, nearly-linear boundary (**high bias**: train and val error both high and
  close together). Large `C` forces the boundary to fit every training point
  (**high variance**: train error → 0 while the val gap widens). The best `C` sits where
  the validation curve bottoms out.
* **Right — error vs number of bagging members.** Averaging more members monotonically
  removes variance, with sharply diminishing returns — the curve flattens onto the bias
  floor, which no amount of bagging can lower. This is the direct payoff of the ensemble.

In [ ]:
# ── Left panel: sweep C (the complexity / regularisation dial) ────────────────
C_SWEEP = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
c_train_err, c_val_err = [], []

print('Sweeping C (single SVM, no bagging — isolates the complexity effect):')
for C in C_SWEEP:
    m = make_svm_pipeline(C=C, gamma=best_gamma, probability=False)
    m.fit(X_train, y_train)
    tr_e = 1 - accuracy_score(y_train_det, m.predict(X_train_det))
    vl_e = 1 - accuracy_score(y_val, m.predict(X_val))
    c_train_err.append(tr_e); c_val_err.append(vl_e)
    print(f'  C={C:<8} train err={tr_e:.4f}   val err={vl_e:.4f}   gap={vl_e-tr_e:+.4f}')

# ── Right panel: cumulative soft-vote over an increasing number of members ────
# Reuses the already-fitted ensemble: averaging the first m members' probabilities is
# exactly what a bagging ensemble of size m predicts.
val_member_probs = member_probas(bagged_svm, X_val)
ens_sizes, ens_err = [], []
running = np.zeros_like(val_member_probs[0])
for m_i, p in enumerate(val_member_probs, start=1):
    running += p
    ens_sizes.append(m_i)
    ens_err.append(1 - accuracy_score(y_val, (running / m_i).argmax(1)))

single_member_err = ens_err[0]
print(f'\nVal error with 1 member  : {single_member_err:.4f}')
print(f'Val error with {len(ens_sizes)} members : {ens_err[-1]:.4f}  '
      f'(variance removed: {single_member_err - ens_err[-1]:+.4f})')

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))

axL.plot(C_SWEEP, c_train_err, 'o-', color='#1f77b4', linewidth=2, label='Train error')
axL.plot(C_SWEEP, c_val_err,   '^-', color='#ff7f0e', linewidth=2, label='Validation error')
axL.fill_between(C_SWEEP, c_train_err, c_val_err, color='#ff7f0e', alpha=0.15,
                 label='Variance (gap)')
axL.axvline(best_C, color='#2ca02c', linestyle='--', linewidth=1.5,
            label=f'Chosen C = {best_C}')
axL.set_xscale('log')
axL.set_xlabel('C  (inverse regularisation → model complexity)', fontsize=11)
axL.set_ylabel('Error rate (1 − accuracy)', fontsize=11)
axL.set_title('SVM Complexity Curve — Under- vs Over-fitting', fontsize=12, fontweight='bold')
axL.legend(fontsize=10); axL.grid(linestyle='--', alpha=0.4)

axR.plot(ens_sizes, ens_err, 'o-', color='#9467bd', linewidth=2,
         label='Bagged ensemble (soft vote)')
axR.axhline(single_member_err, color='#d62728', linestyle='--', linewidth=1.5,
            label=f'Single member ({single_member_err:.3f})')
axR.set_xlabel('Number of bagging members averaged', fontsize=11)
axR.set_ylabel('Validation error rate', fontsize=11)
axR.set_title('Bagging — Variance Falls as Members Are Added',
              fontsize=12, fontweight='bold')
axR.set_xticks(ens_sizes)
axR.legend(fontsize=10); axR.grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('svm_complexity_and_bagging.png', dpi=130, bbox_inches='tight')
plt.show()

## 16. Save Model Artifact

The bagged SVM is pickled with `joblib`. The frozen ResNet18 needs no saving — it is
recreated from torchvision's ImageNet weights, so only the extraction *settings* are
recorded alongside the classifier.

In [ ]:
save_path = OUTPUT_DIR / 'rajuk_svm_bagging.joblib'

artifact = {
    'bagged_svm'        : bagged_svm,          # scaler + PCA + SVC live inside each member
    'model_type'        : 'BaggingClassifier(Pipeline(StandardScaler, PCA, SVC))',
    'feature_extractor' : 'torchvision.models.resnet18(ResNet18_Weights.DEFAULT), fc=Identity, frozen',
    'feature_dim'       : extractor.out_dim,
    'class_names'       : CLASS_NAMES,
    'image_size'        : IMAGE_SIZE,
    'imagenet_mean'     : IMAGENET_MEAN,
    'imagenet_std'      : IMAGENET_STD,
    'label_source'      : str(source_path),
    'label_column'      : level_col,
    'svm_C'             : best_C,
    'svm_gamma'         : best_gamma,
    'svm_kernel'        : SVM_KERNEL,
    'pca_components'    : PCA_COMPONENTS,
    'n_ensemble'        : len(bagged_svm.estimators_),
    'bootstrap_frac'    : BOOTSTRAP_FRAC,
    'max_features_frac' : MAX_FEATURES_FRAC,
    'train_aug_passes'  : TRAIN_AUG_PASSES,
    'tta_enabled'       : TTA_ENABLED,
    'kfold_val_accs'    : fold_val_accs,
    'oob_score'         : getattr(bagged_svm, 'oob_score_', None),
    'train_size'        : int(X_train.shape[0]),
    'val_size'          : int(X_val.shape[0]),
    'test_size'         : int(X_test.shape[0]),
    'val_acc'           : val_acc,
    'final_test_acc'    : test_acc,
}

joblib.dump(artifact, save_path)
print(f'Model saved to  : {save_path}')
print(f'Members         : {len(bagged_svm.estimators_)} SVMs')
print(f'Final test acc  : {test_acc:.4f}')

print('\nTo reuse: extract 512-d ResNet18 features from an image the same way '
      '(resize 224, ImageNet normalise),\nthen call '
      "joblib.load(path)['bagged_svm'].predict_proba(features).")

## 17. Sanity Check — Predict on Random Test Samples

In [ ]:
rng2   = np.random.default_rng(SEED + 1)
sample = rng2.choice(len(y_test), size=min(5, len(y_test)), replace=False)

# Features are already extracted → just index into X_test (order matches test_dataset)
sample_probs = bag_proba(bagged_svm, X_test[sample],
                         X_test_flip[sample] if X_test_flip is not None else None)

fig, axes = plt.subplots(1, len(sample), figsize=(len(sample) * 3.5, 4))
fig.suptitle('Sanity Check — Random Test Predictions (bagged SVM)',
             fontsize=13, fontweight='bold')

for ax, s_idx, probs in zip(np.atleast_1d(axes), sample, sample_probs):
    true_lbl = int(y_test[s_idx])
    pred_lbl = int(probs.argmax())

    ax.imshow(np.array(test_dataset.images[int(s_idx)]))
    ax.set_xticks([]); ax.set_yticks([])

    colour = '#2ca02c' if pred_lbl == true_lbl else '#d62728'
    ax.set_title(
        f'True:  {CLASS_NAMES[true_lbl]}\n'
        f'Pred:  {CLASS_NAMES[pred_lbl]}  ({probs[pred_lbl]:.0%})',
        fontsize=9, color=colour, fontweight='bold'
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(colour); spine.set_linewidth(3)

plt.tight_layout()
plt.savefig('sanity_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

print('Per-class probabilities (soft vote across all members):')
print(f'  {"idx":>5}  ' + '  '.join(f'{c:>10}' for c in CLASS_NAMES) + '   true')
for s_idx, probs in zip(sample, sample_probs):
    print(f'  {int(s_idx):>5}  ' + '  '.join(f'{p:>10.3f}' for p in probs)
          + f'   {CLASS_NAMES[int(y_test[s_idx])]}')